# Projection, Natural Gradient, and Optimal Protocols

### [Neil D. Lawrence](http://inverseprobability.com)

### 2026-11-17

**Abstract**: Maximum entropy as an m-projection onto a constraint
manifold, and natural gradient descent as steepest descent in the Fisher
metric. Geodesics of that metric are Crooks’ minimum-dissipation
protocols. One computed geodesic on a two-parameter exponential family
is enough.

$$
$$

<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

Worksheet 3 is due at the start of this session. No class test. Quiz 3
is 24 November (Fisher metric, natural gradient, dual flatness).

## This Session

**Time plan (120 minutes)**

| Minutes | Block                                            |
|--------:|:-------------------------------------------------|
|    0–10 | Collect Worksheet 3; preview Quiz 3              |
|   10–55 | MaxEnt as projection; dual coordinates           |
|   55–65 | Break                                            |
|  65–100 | Natural gradient; one worked comparison          |
| 100–120 | A geodesic on a two-parameter exponential family |

## MaxEnt as Projection

<!-- SNIPPET: _information/includes/maxent-m-projection.md -->
MaxEnt is the $m$-projection of a reference distribution (often uniform)
onto the constraint surface written in moment coordinates $\eta$. Week
5: $\mathrm{KL}$ is non-negative even when differential entropy is not —
the same $\min\mathrm{KL}(q\|r)$ recipe applies to Jaynes’ die
(discrete) and the Gaussian (continuous). On an exponential family the
constraint surface is a straight line in natural parameters $\theta$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mlai

In [ ]:
faces = np.arange(1, 7)
p_unif = np.ones(6) / 6
# Jaynes die probabilities (mean 4.5)
from scipy.optimize import minimize
def maxent_die(target_mean):
    def objective(p):
        p = np.clip(p, 1e-12, 1); p = p / p.sum()
        return np.sum(p * np.log(p / (np.ones(6)/6)))
    cons = ({'type': 'eq', 'fun': lambda p: np.sum(p)-1},
            {'type': 'eq', 'fun': lambda p: np.dot(p, faces)-target_mean})
    res = minimize(objective, p_unif, constraints=cons)
    return res.x
p_q = maxent_die(4.5)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(faces - 0.15, p_unif, width=0.3, label='reference $r$')
ax.bar(faces + 0.15, p_q, width=0.3, label='MaxEnt $q$')
ax.set_xlabel('face')
ax.set_ylabel('probability')
ax.legend()
ax.set_title('$m$-projection of uniform onto mean=4.5')
mlai.write_figure('m-projection-die.svg', directory='./ml')

<img src="https://mlatcl.github.io/iei/slides/diagrams/ml/m-projection-die.svg" class="" width="75%" style="vertical-align:middle;">

Figure: <i>MaxEnt die as $m$-projection of the uniform reference.</i>

<!-- /SNIPPET: _information/includes/maxent-m-projection.md -->

## Natural Gradient

<!-- SNIPPET: _information/includes/natural-gradient-worked.md -->
Natural gradient ascent $\theta\leftarrow\theta+\eta F^{-1}\nabla L$ is
steepest ascent in the Fisher metric. It removes arbitrary
parameterisation dependence. The same $F$ appears in Crooks’
thermodynamic length.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mlai

In [ ]:
rng = np.random.default_rng(0)
data = rng.normal(2.0, 2.0, 500)

def nll(theta, data):
    mu, sigma2 = theta
    sigma2 = max(sigma2, 1e-6)
    return 0.5 * np.mean((data - mu)**2 / sigma2 + np.log(sigma2))

def fisher_gaussian(mu, sigma2):
    return np.array([[1.0 / sigma2, 0.0], [0.0, 0.5 / sigma2 ** 2]])

def vanilla(theta0, steps=80, eta=0.05):
    th = theta0.copy(); path = [th.copy()]
    for _ in range(steps):
        eps = 1e-4
        g = np.array([(nll(th + eps*np.eye(2)[i], data) - nll(th - eps*np.eye(2)[i], data)) / (2*eps)
                      for i in range(2)])
        th = th - eta * g
        path.append(th.copy())
    return np.array(path)

def natural(theta0, steps=80, eta=0.05):
    th = theta0.copy(); path = [th.copy()]
    for _ in range(steps):
        eps = 1e-4
        g = np.array([(nll(th + eps*np.eye(2)[i], data) - nll(th - eps*np.eye(2)[i], data)) / (2*eps)
                      for i in range(2)])
        F = fisher_gaussian(*th)
        th = th - eta * np.linalg.solve(F, g)
        path.append(th.copy())
    return np.array(path)

path_v = vanilla(np.array([0.0, 1.0]))
path_n = natural(np.array([0.0, 1.0]))
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(path_v[:, 0], path_v[:, 1], 'C0-', label='vanilla')
ax.plot(path_n[:, 0], path_n[:, 1], 'C1-', label='natural')
ax.scatter([2], [4], s=80, c='red', label='target')
ax.set_xlabel('$\\mu$'); ax.set_ylabel('$\\sigma^2$'); ax.legend()
mlai.write_figure('natural-gradient-paths.svg', directory='./ml')

<img src="https://mlatcl.github.io/iei/slides/diagrams/ml/natural-gradient-paths.svg" class="" width="65%" style="vertical-align:middle;">

Figure: <i>Vanilla versus natural gradient paths for Gaussian MLE —
Worksheet 3 core task.</i>

<!-- /SNIPPET: _information/includes/natural-gradient-worked.md -->

## Geodesics as Optimal Protocols

<!-- SNIPPET: _information/includes/geodesic-optimal-protocol.md -->
The geodesic is the prescription for minimum-dissipation protocols under
Crooks’ bound. Natural gradient is the local form of the same
instruction. A straight line in $(\mu,\sigma^2)$ is not generally a
geodesic, but it gives a first length estimate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mlai

In [ ]:
# Compare straight line vs a curved detour in (mu, sigma2)
t = np.linspace(0, 1, 100)
straight = np.column_stack([2*t, 1 + 3*t])
detour = np.column_stack([2*t, 1 + 3*t**2])
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(straight[:,0], straight[:,1], 'k-', linewidth=2, label='straight (Worksheet 3)')
ax.plot(detour[:,0], detour[:,1], 'C1--', linewidth=2, label='detour')
ax.scatter([0,2],[1,4], s=60, c=['green','red'])
ax.set_xlabel('$\\mu$'); ax.set_ylabel('$\\sigma^2$'); ax.legend()
ax.set_title('Two paths — which is shorter in Fisher length?')
mlai.write_figure('geodesic-vs-straight.svg', directory='./ml')

<img src="https://mlatcl.github.io/iei/slides/diagrams/ml/geodesic-vs-straight.svg" class="" width="65%" style="vertical-align:middle;">

Figure: <i>Worksheet 3 uses the straight line; higher marks compare a
second path.</i>

<!-- /SNIPPET: _information/includes/geodesic-optimal-protocol.md -->

## GAIST on Minimum-Dissipation Protocols

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_information/includes/welling-geodesics-and-work.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_information/includes/welling-geodesics-and-work.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

(Welling et al., 2026) has a theory of minimum-dissipation protocols. It
is not this week’s theory. Sections 22.3–22.4 minimise dissipated work
in the Benamou–Brenier formulation of optimal transport. The geodesic
that saturates the bound lives in the *Wasserstein* metric, and the
bound is $\Sigma \ge W_2^2/(T\tau)$. Counterdiabatic driving (Section
22.4) prescribes a path of densities and solves for the control that the
system must follow.

We are still on the equilibrium manifold, with the Fisher metric.
Today’s geodesic is Crooks’ minimum-dissipation protocol; today’s local
instruction is the natural gradient $F^{-1}\nabla L$. GAIST does not
treat m-projection, dual flatness, or the natural gradient. Stay with
Amari this week. Wasserstein and the Schrödinger bridge arrive in week
8, where the three geometries must be kept apart.

## Define This Week

## Named, Not Yet Answered

An alternating (m)-projection onto two constraint sets is how Sinkhorn
enforces two prescribed marginals. Name it; do not compute it. Week 8.

## This Week’s Pair

No-go: you cannot beat (^2/). Prescription: descend by the natural
gradient; travel by the geodesic.

## After This Lecture

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/GITHUB_ORG/iei/edit/gh-pages/_lamd/maxent-projection.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/GITHUB_ORG/iei/edit/gh-pages/_lamd/maxent-projection.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Quiz 3 at the start of lecture 7 (24 November): Fisher metric, natural
gradient, dual flatness, on a new parameterised family.

## Further Reading

-   Sections 22.3–22.4 (Wasserstein geodesics, not Fisher–Rao) of
    Welling et al. (2026)

-   Chapters 3–4 of Amari (2016)

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

## References

Amari, S., 2016. Information geometry and its applications, Applied
mathematical sciences. Springer, Tokyo.
<https://doi.org/10.1007/978-4-431-55978-8>

Welling, M., Lu, S., Holdijk, L., 2026. Generative AI and stochastic
thermodynamics: A tale of free energies. Cambridge University Press,
Cambridge, U.K.